# Dataset characterization — 17 BRGM + 4 public datasets



                                21 CSV datasets
                                      |
                                      v
                              Read and inspect data
                                      |
                                      v
                              Identify timestamps
                              and numeric variables
                                      |
                                      v
                              Sort chronologically
                                      |
                                      v
                                First 50% of data
                                      |
                            +----------+----------+
                            |                     |
                            v                     v
                      Robust STL             ADF test
                      decomposition
                            |
                      +----+----+
                      |    |    |
                      v    v    v
                    Trend Season Residual
                      |    |    |
                      v    v    v
                    Trend Season Noise
                    strength strength ratio
                            |
                            v
                    Convert scores into
                    Low / Medium /
                    Med.-high / High
                            |
                            v
                    Export CSV and LaTeX

In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

DATA_ROOT = PROJECT_ROOT / "Data" / "piezos"
PUBLIC_ROOT = PROJECT_ROOT / "Data" / "Other"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "EDBT_dataset_characterization"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("BRGM folder:", DATA_ROOT, "| exists:", DATA_ROOT.is_dir())
print("Public folder:", PUBLIC_ROOT, "| exists:", PUBLIC_ROOT.is_dir())
print("Output folder:", OUTPUT_ROOT)


## Configurable settings

The BRGM expected columns are `date`, `level`, `PRELIQ_Q`, `T_Q`, `ETP_Q`; its history is 336 steps, horizon 24, and training portion is the first 50% of each chronologically sorted series. The dataset characterization below uses the **training portion for distribution/ADF/STL fitting**, to avoid peeking at test data when reporting properties relevant to model selection. Dataset length, date range, and missingness describe the **complete** CSV.

Seasonality is defined as `max(0, 1 − Var(R) / Var(S + R))`, trend as `max(0, 1 − Var(R) / Var(T + R))`, using robust STL components (`T` trend, `S` seasonal, `R` residual). Scores are clipped to [0, 1]. Noise ratio is `Var(R) / Var(observed)`; this is a **descriptive proxy**, not an estimated physical measurement-error variance. Stationarity: ADF p-value on training *target*; `p < 0.05` rejects the unit-root null; failure to reject **does not prove nonstationarity**. These quantities are target-specific, whereas some literature tables do not specify how they aggregate across variables.


In [ ]:
import pandas as pd
from pathlib import Path

weather_path = PUBLIC_ROOT / "weather" / "weather.csv"

weather = pd.read_csv(weather_path)

weather['date'] = pd.to_datetime(weather['date'])

weather = weather.sort_values('date').reset_index(drop=True)

# Find duplicated timestamps
duplicates = weather[
    weather.duplicated(subset='date', keep=False)
]

print("Duplicated timestamps:")
display(duplicates)

print("Number of duplicated rows:",
      weather['date'].duplicated().sum())

# Check the expected 10-minute frequency
expected_dates = pd.date_range(
    start=weather['date'].min(),
    end=weather['date'].max(),
    freq='10min'
)

missing_dates = expected_dates.difference(
    pd.DatetimeIndex(weather['date'])
)

print("Missing timestamps:", len(missing_dates))
print(missing_dates)

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec('statsmodels') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'])
import re, json, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

PIEZO_NAMES = [f'piezo{i}' for i in list(range(1,10)) + list(range(11,19))]
PUBLIC_NAMES = ['ETTh1','ETTh2','Weather','Electricity']
TARGETS = {'ETTh1':'OT', 'ETTh2':'OT', 'Weather':'T (degC)', 'Electricity':'319'}
# In the common 321-column electricity.csv, customers are named strings "0" ... "320".
# We explicitly select column "319"; this does NOT establish its original MT_320 identity.
# Use the same exact column in the CF forecasting experiment.
# If using a named-meter version of the file, explicitly change this to the intended named column.
DATE_ALIASES = ('date', 'datetime', 'timestamp', 'time')
TRAIN_RATIO = 0.50   # matches the uploaded CF notebook
INPUT_LEN, HORIZON = 336, 24
MAX_STL_TRAIN_POINTS = 6000  # contiguous end of train, for speed; see method note in table
MAX_ADF_TRAIN_POINTS = 5000
# Override season length (number of time steps per cycle), e.g. {'piezo7':365}
SEASONAL_PERIOD_OVERRIDE = {}
# Specify if a data source has no parseable timestamp; only use if definitely known.
FREQUENCY_OVERRIDE = {}  # e.g. {'piezo7':'1 day'}

def strength_label(value):
    if not np.isfinite(value): return 'n/a'
    if value >= 0.70: return 'High'
    if value >= 0.40: return 'Med.-high'
    if value >= 0.20: return 'Medium'
    return 'Low'

def noise_label(noise_ratio):
    if not np.isfinite(noise_ratio): return 'n/a'
    if noise_ratio < 0.10: return 'Low'
    if noise_ratio < 0.30: return 'Medium'
    return 'High'

print('Expected BRGM datasets:', len(PIEZO_NAMES), PIEZO_NAMES)


In [ ]:
import pandas as pd
from pathlib import Path

# Original Weather dataset
# weather_path = PUBLIC_ROOT / "weather.csv"

# Save the corrected version separately
clean_weather_path = PUBLIC_ROOT / "weather/weather_cleaned.csv"

# Load original dataset
weather = pd.read_csv(weather_path)

weather["date"] = pd.to_datetime(weather["date"])

weather = weather.sort_values("date").reset_index(drop=True)

print("Original observations:", len(weather))

# --------------------------------------------------
# STEP 1: Remove identical duplicated observations
# --------------------------------------------------

duplicates = weather[
    weather.duplicated(subset="date", keep=False)
]

if not duplicates.empty:

    # Verify that all duplicate measurements are identical
    for _, group in duplicates.groupby("date"):

        assert (
            group.drop(columns="date").nunique(dropna=False) <= 1
        ).all(), "Conflicting measurements found!"

    weather = weather.drop_duplicates(
        subset="date",
        keep="first"
    )

print("After removing duplicates:", len(weather))


# --------------------------------------------------
# STEP 2: Restore the regular 10-minute time grid
# --------------------------------------------------

weather = weather.set_index("date")

full_index = pd.date_range(
    start=weather.index.min(),
    end=weather.index.max(),
    freq="10min"
)

weather = weather.reindex(full_index)

print("Missing observations before interpolation:",
      weather.isna().all(axis=1).sum())


# --------------------------------------------------
# STEP 3: Interpolate missing numerical measurements
# --------------------------------------------------

numeric_cols = weather.select_dtypes(
    include="number"
).columns

weather[numeric_cols] = (
    weather[numeric_cols]
    .interpolate(
        method="time",
        limit_area="inside"
    )
)

# Restore original column structure
weather.index.name = "date"

weather = weather.reset_index()

# --------------------------------------------------
# STEP 4: Validate corrected dataset
# --------------------------------------------------

assert not weather["date"].duplicated().any()

assert (
    weather["date"].diff().dropna()
    == pd.Timedelta(minutes=10)
).all()

assert not weather[numeric_cols].isna().any().any()

print("\nCorrected observations:", len(weather))
print("Remaining duplicated timestamps:",
      weather["date"].duplicated().sum())

print("Remaining missing values:",
      weather[numeric_cols].isna().sum().sum())


# --------------------------------------------------
# STEP 5: Save without overwriting the original
# --------------------------------------------------

weather.to_csv(
    clean_weather_path,
    index=False
)

print("\nSaved corrected Weather dataset:")
print(clean_weather_path)

## Find files — inspect the list before calculating


In [ ]:
def discover_exact_csvs(root, names):
    if root is None or not Path(root).is_dir(): return {}, list(names), {}
    candidates = list(Path(root).rglob('*.csv'))
    found, dup = {}, {}
    for name in names:
        matches = [p for p in candidates if p.stem.casefold() == name.casefold()]
        if len(matches) == 1: found[name] = matches[0]
        elif len(matches) > 1: dup[name] = [str(p) for p in matches]
    missing = [name for name in names if name not in found and name not in dup]
    return found, missing, dup

piezo_files, missing_piezos, duplicate_piezos = discover_exact_csvs(DATA_ROOT, PIEZO_NAMES)
public_files, missing_public, duplicate_public = discover_exact_csvs(PUBLIC_ROOT, PUBLIC_NAMES)

# Use the corrected Weather dataset for characterization
public_files["Weather"] = clean_weather_path
print("Weather analysis file:", public_files["Weather"])
print('Found BRGM:', len(piezo_files), '/ 17')
display(pd.DataFrame([{'name': k, 'path': str(v)} for k,v in piezo_files.items()]))
print('Missing:', missing_piezos, '\nDuplicates:', duplicate_piezos)
print('Public CSVs found:', list(public_files))
display(pd.DataFrame([{'name': k, 'path': str(v)} for k,v in public_files.items()]))

if duplicate_piezos or duplicate_public:
    raise ValueError('Duplicate dataset CSV stems: resolve paths/duplicates before proceeding.')
if missing_piezos:
    print('WARNING: not all 17 BRGM files found. Check DATA_ROOT; no missing entries will be invented.')
if not piezo_files:
    raise FileNotFoundError('No piezometer CSVs found. Set DATA_ROOT to the folder used by your CF notebook.')

if missing_public:
    raise FileNotFoundError(
        'Missing public CSVs: ' + ', '.join(missing_public) +
        '. Put the four files under PUBLIC_ROOT and rerun. '
        'Expected file stems: ETTh1, ETTh2, Weather, Electricity.'
    )
if len(piezo_files) != 17:
    raise FileNotFoundError(f'Found {len(piezo_files)}/17 BRGM files; fix DATA_ROOT before analysis.')


## Load, inspect timestamps, and estimate train-only time-series properties

No silent imputation of missing rows/targets. The decomposition uses a **contiguous nonmissing final segment of training data**; if the series contains gaps in calendar time or the target has missing values, STL/ADF are skipped and a quality flag is emitted, rather than assuming that consecutive rows mean consecutive time steps. For a short series, annual decomposition may not have enough cycles; the chosen period is exported for review.
**Electricity target selection:** `319` is a numeric column name in the user’s observed `electricity.csv` header. The source dataset contains individual clients, not a canonical predictive target.


In [ ]:
def resolve_date_column(df):
    lookup = {c.casefold():c for c in df.columns}
    for candidate in DATE_ALIASES:
        if candidate in lookup: return lookup[candidate]
    return None

def find_frequency(times, override=None):
    if override is not None: return override, np.nan, np.nan, None
    if times is None: return 'unknown', np.nan, np.nan, None
    good = pd.Series(times).dropna().sort_values()
    diffs = good.diff().dropna()
    diffs = diffs[diffs > pd.Timedelta(0)]
    if diffs.empty: return 'unknown', np.nan, np.nan, None
    step = diffs.mode().iloc[0]
    consistency = float((diffs == step).mean())
    minutes = step.total_seconds()/60
    if minutes < 60: text = f'{minutes:g} min'
    elif minutes < 1440: text = f'{minutes/60:g} hour'
    else: text = f'{minutes/1440:g} day'
    return text, float(minutes), consistency, step

def choose_period(name, interval_minutes, train_len):
    if name in SEASONAL_PERIOD_OVERRIDE:
        return int(SEASONAL_PERIOD_OVERRIDE[name]), 'manual'
    if not np.isfinite(interval_minutes) or interval_minutes <= 0: return None, 'unknown frequency'
    # Hydrological 'daily' is best characterized annually when 2+ years of TRAIN data exist.
    if name.casefold().startswith('piezo') and np.isclose(interval_minutes, 1440) and train_len >= 2*365:
        return 365, 'annual (daily observations)'
    # For hourly and subhourly series choose daily cycle if feasible.
    daily = 1440 / interval_minutes
    if abs(daily-round(daily)) < 1e-6 and 2 <= round(daily) <= train_len//2:
        return int(round(daily)), 'daily'
    weekly = 10080 / interval_minutes
    if abs(weekly-round(weekly)) < 1e-6 and 2 <= round(weekly) <= train_len//2:
        return int(round(weekly)), 'weekly'
    return None, 'no supported cycle'

def clipped_strength(residual, competing):
    den = float(np.var(competing))
    return float(np.clip(1 - np.var(residual)/den, 0, 1)) if den > 0 else np.nan

def analyze_file(name, path, family, target):
    raw = pd.read_csv(path)
    date_col = resolve_date_column(raw)
    if target not in raw.columns:
        raise ValueError(f'{name}: expected target {target!r} not found. Available: {list(raw.columns)}')
    df = raw.copy()
    if date_col:
        parsed = pd.to_datetime(df[date_col], errors='coerce')
        if parsed.isna().any():
            warnings.warn(f'{name}: {parsed.isna().sum()} unparseable timestamps; retained for diagnostic purposes.')
        df['_parsed_date_'] = parsed
        df = df.sort_values('_parsed_date_', kind='stable').reset_index(drop=True)
        dates = df['_parsed_date_']
    else: dates = None
    numeric_cols = [c for c in raw.columns if c != date_col and pd.api.types.is_numeric_dtype(raw[c])]
    nonnumeric_cols = [c for c in raw.columns if c != date_col and c not in numeric_cols]
    n = len(df)
    freq, minutes, regularity, step = find_frequency(dates, FREQUENCY_OVERRIDE.get(name))
    missing_target = int(pd.to_numeric(df[target], errors='coerce').isna().sum())
    missing_any = int(df[numeric_cols].isna().any(axis=1).sum()) if numeric_cols else np.nan
    duplicates = int(dates.duplicated().sum()) if dates is not None else np.nan
    train = df.iloc[:int(n*TRAIN_RATIO)].copy()
    y = pd.to_numeric(train[target], errors='coerce')
    period, rationale = choose_period(name, minutes, len(train))
    notes = []
    if not date_col: notes.append('date column absent')
    if nonnumeric_cols: notes.append('nonnumeric other columns: '+', '.join(nonnumeric_cols))
    if missing_target: notes.append(f'{missing_target} missing target values')
    if dates is not None and dates.isna().any(): notes.append('unparseable dates')
    if np.isfinite(regularity) and regularity < .98: notes.append('irregular timestamps/gaps')
    if dates is not None and duplicates: notes.append(f'{duplicates} duplicated timestamps')
    season = trend = noise = adf_p = np.nan
    usable = y.dropna().to_numpy(dtype=float)
    if period is None or period < 2: notes.append('no defensible STL seasonal period')
    elif len(usable) < 2*period: notes.append('insufficient train points for 2 seasonal cycles')
    elif dates is not None and (not np.isfinite(regularity) or regularity < .98 or duplicates or dates.isna().any()):
        notes.append('STL skipped: irregular/duplicate/unparseable dates')
    elif len(usable) != len(y): notes.append('STL skipped: missing training target')
    elif np.std(usable) <= 0: notes.append('STL skipped: constant target')
    else:
        # Take last contiguous part of TRAIN. Full observed feature count / timespan still use entire CSV.
        block = usable[-max(MAX_STL_TRAIN_POINTS, 2*period):]
        decomposition = STL(block, period=period, seasonal=13, robust=True).fit()
        season = clipped_strength(decomposition.resid, decomposition.seasonal+decomposition.resid)
        trend = clipped_strength(decomposition.resid, decomposition.trend+decomposition.resid)
        noise = float(np.var(decomposition.resid)/np.var(block)) if np.var(block)>0 else np.nan
    if len(usable) >= 50 and np.std(usable)>0 and (dates is None or (np.isfinite(regularity) and regularity >= .98 and not duplicates)):
        try:
            adf_p = float(adfuller(usable[-MAX_ADF_TRAIN_POINTS:], maxlag=12, autolag='AIC')[1])
        except Exception as exc: notes.append('ADF error: '+str(exc)[:80])
    else: notes.append('ADF skipped: missing/short/irregular or constant target')
    result = dict(dataset=name, family=family, source='measured CSV', path=str(path),
        frequency=freq, modal_interval_minutes=minutes, frequency_consistency=regularity,
        variables_M=len(numeric_cols), timestamps_T=n, target=target,
        start_date=str(dates.min())[:19] if dates is not None else '',
        end_date=str(dates.max())[:19] if dates is not None else '',
        missing_target=missing_target, rows_with_missing_numeric=missing_any,
        duplicate_timestamps=duplicates, train_timesteps=len(train),
        STL_period=period, STL_cycle=rationale, seasonality_strength=season, trend_strength=trend,
        noise_ratio=noise, ADF_p=adf_p,
        ADF_reject_5pct='Yes' if np.isfinite(adf_p) and adf_p < .05 else ('No' if np.isfinite(adf_p) else 'n/a'),
        seasonality_label=strength_label(season), trend_label=strength_label(trend), noise_label=noise_label(noise),
        notes='; '.join(notes))
    return result

rows=[]
for name,path in piezo_files.items():
    try: rows.append(analyze_file(name,path,'BRGM','level'))
    except Exception as e: print('ERROR', name, repr(e)); raise
for name, path in public_files.items():
    target = TARGETS[name]
    header = pd.read_csv(path, nrows=0)
    if target not in header.columns:
        raise ValueError(
            f'{name}: target {target!r} absent in {path.name}. '
            f'Available columns: {list(header.columns)}. '
            'Set TARGETS[name] to a column that is actually in YOUR CSV; '
            'do not silently switch to its first series.'
        )
    print(f'{name}: using target column {target!r} from {path.name}')
    rows.append(analyze_file(name, path, 'Public', target))
measured=pd.DataFrame(rows)
pd.set_option('display.max_columns',40)
display(measured[['dataset','frequency','variables_M','timestamps_T','target',
                 'STL_period','seasonality_strength','trend_strength','noise_ratio','ADF_p','notes']])
measured.to_csv(OUTPUT_ROOT/'dataset_properties_measured.csv',index=False)
print('Saved:', OUTPUT_ROOT/'dataset_properties_measured.csv')


## All four public CSVs must be measured

**Interpretation:** ETT/Weather/Electricity use a daily STL period (24, 144, 24 observations respectively, if the actual timestamps confirm their standard frequencies); daily BRGM uses an annual 365-observation period. Thus numerical strengths share definitions but measure **different seasonal cycles**. For Electricity, the scores describe only client `MT_320`, while $M=321$ describes the full input table. Analysing a target **does not** prove that other clients are useful/actionable predictors.


In [ ]:
if len(measured) != len(PIEZO_NAMES) + len(PUBLIC_NAMES):
    raise RuntimeError('Not all 21 datasets were measured.')
combined = measured.copy()
combined['sort_index'] = combined['dataset'].map(
    {s: i for i, s in enumerate(PIEZO_NAMES + PUBLIC_NAMES)}
)
combined = combined.sort_values('sort_index').drop(columns='sort_index').reset_index(drop=True)
combined.to_csv(OUTPUT_ROOT / 'dataset_properties_with_screenshot_references.csv', index=False)
print('BRGM measured:', sum(combined.family == 'BRGM'))
print('Public measured:', sum(combined.family == 'Public'))
print('Screenshot placeholders: 0')
display(combined[['dataset', 'source', 'target', 'frequency',
                  'variables_M', 'timestamps_T', 'STL_period',
                  'seasonality_label', 'trend_label', 'noise_label',
                  'ADF_reject_5pct', 'notes']])


## Export LaTeX tables and a methodology note




In [ ]:
def latex_escape(value):
    value = str(value)
    return (value.replace('\\', r'\textbackslash{}')
            .replace('&', r'\&').replace('%', r'\%')
            .replace('_', r'\_').replace('#', r'\#'))

def nice_range(vals, formatter=lambda x: f'{int(x):,}'):
    vals = pd.to_numeric(pd.Series(vals), errors='coerce').dropna()
    if vals.empty: return '--'
    lo, hi = vals.min(), vals.max()
    return formatter(lo) if lo == hi else formatter(lo) + '--' + formatter(hi)

def fmt_num(x, spec='.2f'):
    x = pd.to_numeric(pd.Series([x]), errors='coerce').iloc[0]
    return format(x, spec) if np.isfinite(x) else '--'

b = combined[combined.family == 'BRGM']
if len(b) != 17: raise ValueError('Expected exactly 17 BRGM measurements')
freqs = b.frequency.dropna().astype(str).unique()
summary = {
    'frequency': freqs[0] if len(freqs) == 1 else 'varies',
    'M': nice_range(b.variables_M),
    'T': nice_range(b.timestamps_T),
    'season': fmt_num(b.seasonality_strength.median()),
    'trend': fmt_num(b.trend_strength.median()),
    'noise': fmt_num(b.noise_ratio.median()),
}
mainlines = [
    r'\begin{table*}[t]',
    r'\centering',
    r'\caption{Characteristics of the 17 BRGM piezometric series and four public multivariate forecasting datasets. BRGM statistics summarize the 17 separate series; all seasonality, trend and residual-noise scores are computed on the chosen training target.}',
    r'\label{tab:dataset_characteristics}',
    r'\footnotesize',
    r'\setlength{\tabcolsep}{4pt}',
    r'\begin{tabular}{@{}lcccccc@{}}',
    r'\toprule',
    r'Dataset & Sampling & $M$ & $T$ & Seasonality & Trend & Residual ratio \\',
    r'\midrule',
]
mainlines.append(
    f"BRGM piezometers ($n=17$) & {latex_escape(summary['frequency'])} & "
    f"{summary['M']} & {summary['T']} & {summary['season']} & "
    f"{summary['trend']} & {summary['noise']} " + r'\\'
)
mainlines.append(r'\midrule')
for name in PUBLIC_NAMES:
    r = combined.loc[combined.dataset == name].iloc[0]
    mainlines.append(
        f"{latex_escape(name)} & {latex_escape(r.frequency)} & "
        f"{int(r.variables_M):,} & {int(r.timestamps_T):,} & "
        f"{fmt_num(r.seasonality_strength)} & {fmt_num(r.trend_strength)} & "
        f"{fmt_num(r.noise_ratio)} " + r'\\'
    )
mainlines += [
    r'\bottomrule',r'\end{tabular}',r'\vspace{1mm}',
    r'\parbox{0.98\textwidth}{\scriptsize \textit{Notes.} $M$: numeric variables excluding date; $T$: full-series observations. BRGM $M$/$T$ are ranges and reported STL scores are medians. STL is fitted on the last at most 6,000 observations of the first 50\% of each target series; ADF uses at most 5,000 training observations. BRGM uses an annual STL cycle (365 days); the public series use daily cycles determined from their recorded timestamps. Residual ratio is $\mathrm{Var}(R)/\mathrm{Var}(y)$, not directly observed physical measurement noise. Electricity scores concern target column 319 in the anonymized benchmark CSV, while $M=321$ describes all client series. The original meter identifier for this column has not been verified.}',
    r'\end{table*}'
]
maintex='\n'.join(mainlines)+'\n'
(OUTPUT_ROOT/'main_dataset_table.tex').write_text(maintex,encoding='utf8')

fulllines = [
    r'\begin{table*}[t]',r'\centering',
    r'\caption{Individual time-series dataset characteristics computed using a common target-specific train-only protocol.}',
    r'\label{tab:dataset_characteristics_full}',r'\scriptsize',
    r'\setlength{\tabcolsep}{3pt}',
    r'\begin{tabular}{@{}llrrccccc@{}}',r'\toprule',
    r'Dataset & Sampling & $M$ & $T$ & Seasonality & Trend & Noise & ADF $p$ & Target \\',
    r'\midrule'
]
for _, r in combined.iterrows():
    fulllines.append(
        f"{latex_escape(r.dataset)} & {latex_escape(r.frequency)} & "
        f"{int(r.variables_M):,} & {int(r.timestamps_T):,} & "
        f"{latex_escape(r.seasonality_label)} & {latex_escape(r.trend_label)} & "
        f"{latex_escape(r.noise_label)} & {fmt_num(r.ADF_p,'.3g')} & "
        f"{latex_escape(r.target)} " + r'\\'
    )
    if r.dataset == PIEZO_NAMES[-1]: fulllines.append(r'\midrule')
fulllines += [r'\bottomrule',r'\end{tabular}',r'\vspace{1mm}',
    r'\parbox{0.98\textwidth}{\scriptsize \textit{Notes.} Seasonality/trend strengths: Low $<0.20$, Medium $[0.20,0.40)$, Med.--high $[0.40,0.70)$, High $\geq0.70$. Residual-to-target variance ratio: Low $<0.10$, Medium $[0.10,0.30)$, High $\geq0.30$. Seasonal periods are annual for daily BRGM and daily for the public datasets. ADF $p$ tests the unit-root null and does not establish stationarity or nonstationarity on its own.}',
    r'\end{table*}']
fulltex='\n'.join(fulllines)+'\n'
(OUTPUT_ROOT/'full_dataset_table.tex').write_text(fulltex,encoding='utf8')
methods_note = (
    'All 21 rows calculated directly from CSV files. Train-only target statistics: '
    'STL period chosen from observed timestamps (365 for daily BRGM, 24 for hourly '
    'ETTh1, ETTh2, Electricity, 144 for 10-minute Weather when frequencies match); '
    'last up to 6000 training observations, robust=True, seasonal=13. '
    'Seasonality strength=max(0,1-Var(R)/Var(S+R)); '
    'trend strength=max(0,1-Var(R)/Var(T+R)); '
    'noise proxy=Var(R)/Var(target); ADF p from last <=5000 training observations '
    '(maxlag=12, autolag=AIC). Train is first 50% of chronologically sorted series. '
    'Full-series M,T and frequency obtained from the CSV. '
    'Electricity column 319 is the explicitly chosen target in the anonymized CSV, not a canonical '
    'target or a validated causal outcome; mapping to original MT_320 is unverified. M=321 counts all clients. '
)
(OUTPUT_ROOT/'dataset_methods_note.txt').write_text(methods_note,encoding='utf8')
print('Created:',OUTPUT_ROOT/'main_dataset_table.tex')
print('Created:',OUTPUT_ROOT/'full_dataset_table.tex')
print('Created:',OUTPUT_ROOT/'dataset_methods_note.txt')
print('\n--- MAIN TABLE PREVIEW ---\n', maintex)
